# ЛР4: GQA, KV-cache и инференс

Ноутбук рассчитан на Google Colab. Рекомендуемый runtime: `T4 GPU`.

Что делает notebook:
- клонирует ветку `lab4`;
- ставит зависимости;
- готовит `.env` с `ROOT_DIR`;
- готовит Wikitext, BPE tokenizer и packed dataset;
- проверяет GQA и KV-cache tests;
- обучает GQA-модель;
- показывает TensorBoard;
- генерирует текст из checkpoint с KV-cache;
- сохраняет checkpoint/logs в Google Drive.


## 1. Загрузка проекта


In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/BogdanRoshchupkin/modern-ai-architecture.git"
BRANCH = "lab4"
REPO_DIR = Path("/content/modern-ai-architecture")

if not (REPO_DIR / ".git").exists():
    if REPO_DIR.exists():
        raise RuntimeError(f"{REPO_DIR} exists, but it is not a git repository")
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)], check=True)
else:
    os.chdir(REPO_DIR)
    subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "checkout", BRANCH], check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)

os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())
print("Active branch:", subprocess.check_output(["git", "branch", "--show-current"], text=True).strip())
print("Latest commit:", subprocess.check_output(["git", "log", "--oneline", "-1"], text=True).strip())


## 2. Установка зависимостей


In [ ]:
import sys
import subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Dependencies are ready")


## 3. Проверка GPU и создание `.env`


In [ ]:
from pathlib import Path
import subprocess
import torch

try:
    print(subprocess.check_output(["nvidia-smi"], text=True))
except Exception as exc:
    print("nvidia-smi is not available:", exc)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Path(".env").write_text(f"ROOT_DIR={Path.cwd()}", encoding="utf-8")
print(Path(".env").read_text())


## 4. Подготовка данных

Для Colab готовим компактный pipeline на Wikitext: clean records -> BPE tokenizer -> packed dataset. Если файлы уже есть, ячейка их не пересоздает.


In [ ]:
from pathlib import Path
import subprocess
import sys

processed = Path("data/processed")
processed.mkdir(parents=True, exist_ok=True)

if not Path("data/processed/wikitext_clean.jsonl").exists():
    subprocess.run([
        sys.executable, "-m", "cli.lab1", "prepare-wikitext",
        "--limit", "20000",
        "--raw-output", "data/processed/wikitext_raw.jsonl",
        "--clean-output", "data/processed/wikitext_clean.jsonl",
        "--min-words", "5",
    ], check=True)
else:
    print("wikitext_clean.jsonl already exists")

if not Path("data/processed/common_crawl_bpe.json").exists():
    subprocess.run([
        sys.executable, "-m", "cli.lab1", "tokenize",
        "--input", "data/processed/wikitext_clean.jsonl",
        "--bpe-vocab-size", "1000",
        "--bpe-output", "data/processed/common_crawl_bpe.json",
        "--bpe-train-limit", "20000",
    ], check=True)
else:
    print("common_crawl_bpe.json already exists")

if not Path("data/processed/wikitext_packed.jsonl").exists():
    subprocess.run([
        sys.executable, "-m", "cli.lab1", "pack",
        "--input", "data/processed/wikitext_clean.jsonl",
        "--tokenizer", "data/processed/common_crawl_bpe.json",
        "--output", "data/processed/wikitext_packed.jsonl",
        "--max-length", "512",
    ], check=True)
else:
    print("wikitext_packed.jsonl already exists")


## 5. Проверка GQA и KV-cache тестами


In [ ]:
!python -m pytest tests/test_gqa_kv_cache.py -q


## 6. Быстрая проверка train loop

Эта ячейка запускает один batch и нужна перед полноценным обучением.


In [ ]:
!python -m cli.lab4 train --config configs/lab4_gqa.yaml --fast-dev-run


## 7. Полное обучение

Цель PDF: обучить модель и получить validation perplexity. Для полного балла нужен `val_perplexity <= 25`, для частичного `<= 40`. На T4 этот запуск может занять время.


In [ ]:
!python -m cli.lab4 train --config configs/lab4_gqa.yaml


## 8. TensorBoard


In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/tensorboard


## 9. Найти лучший checkpoint


In [ ]:
from pathlib import Path

ckpts = sorted(Path("checkpoints/lab4_gqa").glob("final*.ckpt"))
print("Found checkpoints:")
for ckpt in ckpts:
    print(ckpt)
assert ckpts, "No final checkpoint found. Run training first."
BEST_CKPT = str(ckpts[-1])
print("BEST_CKPT=", BEST_CKPT)


## 10. Генерация с KV-cache


In [ ]:
prompt = "The history of artificial intelligence"
!python -m cli.lab4 generate --config configs/lab4_gqa.yaml --checkpoint "$BEST_CKPT" --prompt "$prompt" --max-new-tokens 80


## 11. Генерация без KV-cache для сравнения


In [ ]:
prompt = "The history of artificial intelligence"
!python -m cli.lab4 generate --config configs/lab4_gqa.yaml --checkpoint "$BEST_CKPT" --prompt "$prompt" --max-new-tokens 80 --no-kv-cache


## 12. Сохранение результатов в Google Drive

Эта ячейка монтирует Drive и копирует checkpoints, TensorBoard logs и конфиг. Можно пропустить, если Drive не нужен.


In [ ]:
from pathlib import Path
import shutil

try:
    from google.colab import drive
    drive.mount('/content/drive')
    target = Path('/content/drive/MyDrive/modern-ai-architecture-lab4')
    target.mkdir(parents=True, exist_ok=True)
    for source in [Path('checkpoints/lab4_gqa'), Path('logs/tensorboard'), Path('configs/lab4_gqa.yaml')]:
        destination = target / source.name
        if source.is_dir():
            if destination.exists():
                shutil.rmtree(destination)
            shutil.copytree(source, destination)
        elif source.exists():
            shutil.copy2(source, destination)
    print('Saved artifacts to', target)
except Exception as exc:
    print('Drive save skipped:', exc)


## 13. Что показать на защите

- В конфиге `n_heads=4`, `n_kv_heads=2`: это GQA.
- В тестах `KV-cache incremental logits == full forward logits`.
- В training output/TensorBoard показать `val_perplexity`.
- В generation output показать продолжение текста из checkpoint.
- Если нужно, показать генерацию с `--no-kv-cache` и без него.
